# RoadRisk AI - Exposure-Normalised Hotspot Prototype

**Purpose:** Turn RoadRisk event records into a dynamic road-risk map and proposal-ready graphs.

This notebook uses **two inputs**:

1. `roadrisk_events.csv` - near-miss / conflict events.
2. `roadrisk_exposure.csv` - how often each road segment was observed.

Keeping event counts separate from exposure prevents a busy road from looking dangerous simply because it was observed more often.

> The RoadRisk score in this notebook is a **prototype index**, not a calibrated crash probability. The weights must later be validated using real fleet data.

## 1. Setup

In [ ]:
!pip -q install folium

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
from google.colab import files
from IPython.display import display

OUTPUT_DIR='/content/roadrisk_hotspot_outputs'
os.makedirs(OUTPUT_DIR,exist_ok=True)
plt.rcParams['figure.figsize']=(9,5.5)
plt.rcParams['axes.grid']=True
plt.rcParams['grid.alpha']=0.2


## 2. Upload event data

Required columns:

`event_id, timestamp, latitude, longitude, road_segment, event_type, severity, confidence`

Optional columns:

`ttc_seconds, pet_seconds, hard_brake, speed_kmh`


In [ ]:
print('Upload roadrisk_events.csv')
uploaded=files.upload()
events_file=next(iter(uploaded))
events=pd.read_csv(events_file)
events.columns=[str(c).strip() for c in events.columns]

display(events.head())
print('Rows:',len(events))
print('Columns:',list(events.columns))


## 3. Upload exposure data

Recommended columns:

`road_segment, time_bucket, passes, vehicle_km`

Use the same `road_segment` names as the events file. `passes` is the minimum useful exposure measure; `vehicle_km` is optional.


In [ ]:
print('Upload roadrisk_exposure.csv')
uploaded_exp=files.upload()
exposure_file=next(iter(uploaded_exp))
exposure=pd.read_csv(exposure_file)
exposure.columns=[str(c).strip() for c in exposure.columns]

display(exposure.head())
print('Rows:',len(exposure))
print('Columns:',list(exposure.columns))


## 4. Clean and score events

The default weights below are **demonstration values only**. They make the prototype explainable while you collect evidence for calibration.

In [ ]:
ev=events.copy()
ev['timestamp']=pd.to_datetime(ev['timestamp'],errors='coerce')
ev['confidence']=pd.to_numeric(ev.get('confidence',1.0),errors='coerce').fillna(1.0).clip(0,1)

ev['severity_norm']=ev['severity'].astype(str).str.strip().str.lower()
ev['event_type_norm']=ev['event_type'].astype(str).str.strip().str.lower().str.replace(' ','_',regex=False)

severity_weight={'low':1.0,'medium':2.0,'high':4.0,'critical':6.0}
event_weight={
    'hard_brake':0.8,
    'cut_in':1.0,
    'unsafe_overtake':1.2,
    'pedestrian_conflict':1.3,
    'near_collision':1.5,
    'near_miss':1.5,
}

ev['severity_weight']=ev['severity_norm'].map(severity_weight).fillna(1.0)
ev['event_weight']=ev['event_type_norm'].map(event_weight).fillna(1.0)
ev['weighted_event_score']=ev['severity_weight']*ev['event_weight']*ev['confidence']

# Create a coarse time bucket if one is not provided in event data.
ev['hour']=ev['timestamp'].dt.hour
def period(h):
    if pd.isna(h): return 'Unknown'
    if 5 <= h < 10: return 'Morning peak'
    if 10 <= h < 16: return 'Midday'
    if 16 <= h < 20: return 'Evening peak'
    return 'Night / off-peak'
ev['time_bucket']=ev['hour'].apply(period)

display(ev[['event_id','road_segment','event_type','severity','confidence','weighted_event_score','time_bucket']].head())


## 5. Calculate exposure-normalised road risk

Prototype formula:

\[
RiskRate_{s,t} = 1000 	imes rac{\sum WeightedEventScore}{ObservedPasses}
\]

A small smoothing constant is used only to avoid division by zero; segments with no exposure should be excluded rather than interpreted.

In [ ]:
exp=exposure.copy()
exp['passes']=pd.to_numeric(exp['passes'],errors='coerce')
if 'time_bucket' not in exp.columns:
    exp['time_bucket']='All'

# Aggregate event evidence by road segment and time bucket.
event_agg=(ev.groupby(['road_segment','time_bucket'],dropna=False)
             .agg(event_count=('event_id','count'),
                  weighted_events=('weighted_event_score','sum'),
                  mean_confidence=('confidence','mean'))
             .reset_index())

# Aggregate exposure in case the file contains multiple rows per segment/time bucket.
exp_agg=(exp.groupby(['road_segment','time_bucket'],dropna=False)
          .agg(passes=('passes','sum'),
               vehicle_km=('vehicle_km','sum') if 'vehicle_km' in exp.columns else ('passes','size'))
          .reset_index())

risk=event_agg.merge(exp_agg,on=['road_segment','time_bucket'],how='left')
risk['passes']=risk['passes'].fillna(0)
risk['risk_per_1000_passes']=np.where(risk['passes']>0,
    1000*risk['weighted_events']/risk['passes'], np.nan)

# Normalise observed rates into a 0-100 dashboard index.
valid=risk['risk_per_1000_passes'].dropna()
if len(valid) and valid.max()>valid.min():
    risk['RoadRisk_index']=100*(risk['risk_per_1000_passes']-valid.min())/(valid.max()-valid.min())
else:
    risk['RoadRisk_index']=0.0

risk=risk.sort_values('RoadRisk_index',ascending=False)
display(risk.head(20))
risk.to_csv(os.path.join(OUTPUT_DIR,'roadrisk_segment_scores.csv'),index=False)


### Figure 7 - Highest-risk road segments

In [ ]:
segment=(risk.groupby('road_segment')
           .agg(weighted_events=('weighted_events','sum'),passes=('passes','sum'))
           .reset_index())
segment['risk_per_1000_passes']=np.where(segment['passes']>0,1000*segment['weighted_events']/segment['passes'],np.nan)
segment=segment.dropna().sort_values('risk_per_1000_passes',ascending=False)

top=segment.head(12).sort_values('risk_per_1000_passes')
ax=top.set_index('road_segment')['risk_per_1000_passes'].plot(kind='barh',figsize=(10,6.5))
ax.set_title('Prototype RoadRisk Rate by Road Segment')
ax.set_xlabel('Weighted safety events per 1,000 observed passes')
ax.set_ylabel('Road segment')
plt.tight_layout()
path=os.path.join(OUTPUT_DIR,'fig_07_segment_risk.png')
plt.savefig(path,dpi=220,bbox_inches='tight')
plt.show()
print('Saved:',path)


### Figure 8 - Risk by time of day

In [ ]:
time_risk=(risk.groupby('time_bucket')
             .agg(weighted_events=('weighted_events','sum'),passes=('passes','sum'))
             .reset_index())
time_risk['risk_per_1000_passes']=np.where(time_risk['passes']>0,1000*time_risk['weighted_events']/time_risk['passes'],np.nan)

ax=time_risk.set_index('time_bucket')['risk_per_1000_passes'].plot(kind='bar')
ax.set_title('Prototype RoadRisk Rate by Time Period')
ax.set_xlabel('Time period')
ax.set_ylabel('Weighted safety events per 1,000 observed passes')
ax.tick_params(axis='x',rotation=20)
plt.tight_layout()
path=os.path.join(OUTPUT_DIR,'fig_08_time_risk.png')
plt.savefig(path,dpi=220,bbox_inches='tight')
plt.show()
print('Saved:',path)


### Figure 9 - Interactive hotspot map

The map uses mean event coordinates by road segment. It is useful for an MVP demo, but a production system should map-match events to a road network such as OpenStreetMap rather than treating raw GPS points as perfect.

In [ ]:
# Ensure numeric coordinates and remove invalid rows.
ev['latitude']=pd.to_numeric(ev['latitude'],errors='coerce')
ev['longitude']=pd.to_numeric(ev['longitude'],errors='coerce')
coords=ev.dropna(subset=['latitude','longitude']).copy()

if len(coords)==0:
    print('No valid coordinates available; map skipped.')
else:
    segment_coords=(coords.groupby('road_segment')
                    .agg(latitude=('latitude','mean'),longitude=('longitude','mean'))
                    .reset_index())
    map_df=segment.merge(segment_coords,on='road_segment',how='left').dropna(subset=['latitude','longitude'])
    center=[map_df['latitude'].mean(),map_df['longitude'].mean()]
    m=folium.Map(location=center,zoom_start=12,tiles='OpenStreetMap')

    # Circle size encodes the prototype risk rate. We intentionally keep default styling simple.
    max_r=max(map_df['risk_per_1000_passes'].max(),1e-9)
    for _,r in map_df.iterrows():
        radius=5+18*(r['risk_per_1000_passes']/max_r)
        folium.CircleMarker(
            location=[r['latitude'],r['longitude']],
            radius=radius,
            tooltip=f"{r['road_segment']} | Risk rate: {r['risk_per_1000_passes']:.2f}",
            popup=f"{r['road_segment']}<br>Weighted events/1000 passes: {r['risk_per_1000_passes']:.2f}"
        ).add_to(m)

    map_path=os.path.join(OUTPUT_DIR,'fig_09_roadrisk_map.html')
    m.save(map_path)
    display(m)
    print('Saved:',map_path)


## 6. Export outputs

In [ ]:
zip_path='/content/roadrisk_hotspot_outputs.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        z.write(os.path.join(OUTPUT_DIR,fname),arcname=fname)
print('Created:',zip_path)
files.download(zip_path)


## What to say in the proposal

Do **not** say that a road with the highest prototype score is proven to have the highest crash probability.

Instead say:

> RoadRisk proposes to aggregate safety-critical events by road segment and time, then normalise those observations by road exposure. This avoids the common bias of labelling a busy road as dangerous simply because it produces more raw events. The prototype score will be calibrated and validated during fleet pilots before being used for operational routing decisions.
